# MEPI v1 — compare completed pretraining runs

**Purpose:** read measured COMPLETE evidence and select strictly by minimum validation MAE_norm.  
**Inputs:** eight `experiments/pretrain_v1/<backbone>/run_evidence.json` files.  
**Outputs:** comparison CSV/Markdown and the measured selection JSON.  
**Frozen protocol:** MEPI-FROZEN-PROTOCOL v1.1.  
**Trains a model:** no.  
**Checkpoints/results:** reads checkpoints only by recorded hash; writes reports and `artifacts/manuscript/selected_pretraining_backbone.json`.

In [1]:
from pathlib import Path

# MEPI Run-All configuration. Safe recovery is the default.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "mepi_v1").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
SEED = 42
FORCE_RETRAIN = False
AUTO_RESUME = True
CHECKPOINT_EVERY_N_STEPS = 1000

print(f"[PASS] Configuration loaded: PROJECT_ROOT={PROJECT_ROOT}, SEED={SEED}")
print(f"FORCE_RETRAIN={FORCE_RETRAIN}; AUTO_RESUME={AUTO_RESUME}; checkpoint interval={CHECKPOINT_EVERY_N_STEPS} steps")


[PASS] Configuration loaded: PROJECT_ROOT=<REPOSITORY_ROOT>, SEED=42
FORCE_RETRAIN=False; AUTO_RESUME=True; checkpoint interval=1000 steps


## Load read-only comparison workflow

The comparison reads evidence only and never imports or invokes a training loop.

In [2]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.mepi_v1.notebook_workflow import compare_completed_pretraining
print("[PASS] Project source imports completed")


[PASS] Project source imports completed


## Compare measured COMPLETE runs

Pending backbones remain visibly pending; completed rows sort by validation MAE_norm ascending.

In [3]:
comparison_rows = compare_completed_pretraining(PROJECT_ROOT)


Completed runs: 8/8
Validation-only selection: xLSTM (MAE_norm=0.015028174539595075)


## Final comparison table

Display validation and test metrics separately. The winner is never hard-coded.

In [4]:
import pandas as pd
comparison = pd.DataFrame(comparison_rows)
display(comparison)
completed = comparison[comparison['status'] == 'COMPLETE'] if not comparison.empty else comparison
if completed.empty:
    print('COMPARISON STATUS: PENDING — no measured COMPLETE runs')
else:
    print(f"VALIDATION-SELECTED BACKBONE: {completed.iloc[0]['backbone']}")
print('Comparison artifacts written under reports/; selection JSON is measured-only.')


,backbone,status,parameter_count,best_epoch,validation_mae_norm,validation_rmse,validation_r2,test_mae_norm,test_rmse,test_r2,checkpoint_sha256,resumed
0,xLSTM,COMPLETE,3336202,20,0.015028,0.021306,0.999554,0.015226,0.021980,0.999525,6ad61ea11ec661e0633cc0e23dfdb52f5904b5fb2a6287...,False
1,LSTM-Attention,COMPLETE,2547210,20,0.016743,0.023124,0.999475,0.016856,0.023322,0.999465,9869fb82a0f2226aeb50c9ddc08dae7aa3b9b95690fe35...,False
2,BiGRU,COMPLETE,1888266,19,0.018385,0.025944,0.999339,0.018333,0.025763,0.999347,07baf54051118f9f4a0f1eaff09f7ac30451f05851a097...,False
3,LSTM,COMPLETE,2807818,17,0.019087,0.026984,0.999285,0.018894,0.026912,0.999288,8b47bafc931e15f46f17de0876c08bf2424debf6f66429...,False
4,GRU,COMPLETE,2281482,18,0.021471,0.029418,0.999150,0.021350,0.029472,0.999146,56f0a7e0adcb9bde2f8c8c4d507f84743afd3437c6cabb...,False
5,BiLSTM,COMPLETE,2283530,16,0.023646,0.030492,0.999087,0.023369,0.030212,0.999102,22c25505690b69bf545cdd9ed90febf2719acce4e98a17...,False
6,TCN,COMPLETE,2279434,19,0.025892,0.035619,0.998754,0.025828,0.035373,0.998770,da871a3a908a65a349a8c2dfe901269e545a000a8df7ce...,False
7,RWKV,COMPLETE,3857418,19,0.026498,0.038218,0.998566,0.026437,0.037857,0.998591,805294d5d1f3b7b76290d4dce2bdf8e25e8cd1db79d8ec...,False


VALIDATION-SELECTED BACKBONE: xLSTM
Comparison artifacts written under reports/; selection JSON is measured-only.
